<a href="https://colab.research.google.com/github/swartzbt/InverseDesignWorkshop_AISS26/blob/main/InverseDesignHologram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inverse Design with PyTorch

**Duration:** 45 minutes

Design a simple diffractive phase hologram using PyTorch's automatic differentiation.

## 1. Environment Setup

In this workshop we'll use PyTorch as an engineering design tool.

**Objectives**
- Use autograd
- Build a custom `torch.nn.Module`
- Define a custom merit function
- Optimize a physical system

Run the following cell to import dependencies.

In [ ]:
import os

import torch
import torch.nn as nn
import torch.fft as fft
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib import ticker, colors
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
from PIL import Image, ImageOps
from google.colab import files

if not os.path.exists("goat.jpg"):
    !wget -q https://raw.githubusercontent.com/swartzbt/InverseDesignWorkshop_AISS26/main/goat.jpg

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## 2. Autograd

We'll verify autograd on a simple differentiable function.

I've chosen a simple polynomial as an example:

`f(x, y) = (1 - x)**2 + (y - x)**2`

Use autograd to calculate the gradient, and compare with the analytical result.

In [ ]:
# Define a function to differentiate.
def f(x, y):
    return (1 - x)**2 + (y - x)**2

# Define point to calculate
x = torch.tensor(0.5, requires_grad=True)  # Variables with requires_grad=True will have their gradients tracked
y = torch.tensor(1.0, requires_grad=True)

# Calculate the gradients using autograd
z = f(x, y)
z.backward()                 # Calculates the gradients of z with respect to each tracked input
print("Autograd:")
print("df/dx =", x.grad.item())  # x.grad is how you access the gradient of the output with respect to x.
print("df/dy =", y.grad.item())

# Compare with analytical result
def df_dx(x, y):
    return -2 * (1 - x) - 2 * (y - x)

def df_dy(x, y):
    return 2 * (y - x)

print("Analytical:")
print("df/dx =", df_dx(x, y).item())
print("df/dy =", df_dy(x, y).item())

### 2.1. Visualize function and gradient descent step
Plot the above function f(x, y) and show the next step of gradient descent.

**Look at the plot. Have we chosen a good value for the learning rate? How do you know?**

In [ ]:
learn_rate = 0.5

# Create a grid for plotting
x_plot = torch.linspace(0., 2., 300)
y_plot = torch.linspace(0., 2., 300)
X, Y = torch.meshgrid(x_plot, y_plot, indexing="xy")

# Evaluate the function over the grid
Z = f(X, Y)

# Plot contours
plt.figure(figsize=(6, 5))
plt.contourf(X, Y, Z, levels=30)
plt.axis('square')
plt.colorbar()

# Mark the global minimum
plt.plot(1, 1, "r*", markersize=14, label="Minimum")

# Mark the point where we calculated the gradient
plt.plot(x.item(), y.item(), "wo", markersize=7)

# Draw a gradient descent step with the learn rate from the top of the section
plt.arrow(
    x.item(), y.item(),
    -learn_rate * x.grad.item(),
    -learn_rate * y.grad.item(),
    width=0.015, color="white", length_includes_head=True
)

plt.xlabel("x")
plt.ylabel("y")
plt.title("Function and Gradient Descent Step")
plt.legend()
plt.show()

## 3. Gradient Descent

Use gradient descent to find the minimum of the function defined in the previous section.

In [ ]:
# Define starting point
x = torch.tensor(0.5, requires_grad=True)
y = torch.tensor(1.0, requires_grad=True)

# Choose learning rate
learn_rate = 0.5

# Store optimization history
x_history = [x.item()]
y_history = [y.item()]
f_history = [f(x, y).item()]

# Iteratively calculate the gradients and update x and y
for i in range(10):
    z = f(x, y)
    z.backward()

    with torch.no_grad():
        x -= learn_rate * x.grad
        y -= learn_rate * y.grad

    x.grad.zero_()
    y.grad.zero_()

    # Save the new values
    x_history.append(x.item())
    y_history.append(y.item())
    f_history.append(f(x, y).item())

# ----- Plot Results -----
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

# ----- Plot 1: Gradient descent path -----
contour = ax[0].contourf(X, Y, Z, levels=30)

# Mark global minimum
ax[0].plot(1, 1, "r*", markersize=14, label="Minimum")

# Draw each update step
colors = ["white", "cyan"]

for i in range(10):
    ax[0].arrow(
        x_history[i], y_history[i],
        x_history[i+1] - x_history[i],
        y_history[i+1] - y_history[i],
        width=0.012, color=colors[i % 2], length_includes_head=True
    )

ax[0].set_xlabel("x")
ax[0].set_ylabel("y")
ax[0].set_title("Gradient Descent")
ax[0].set_aspect("equal")
fig.colorbar(contour, ax=ax[0])
ax[0].legend()

# ----- Plot 2: Function value -----
ax[1].plot(f_history, "o-")

ax[1].set_xlabel("Iteration")
ax[1].set_ylabel("f(x, y)")
ax[1].set_title("Function Value")
ax[1].grid()

plt.tight_layout()
plt.show()

Pretty simple, right?

This basic framework is used for all gradient descent in PyTorch, whether we are training an AI model or engineering an optimized system.

We define a model we want to train, then a loss (or merit) function that quantifies how good the model is doing, and finally use gradient descent (or ascent) to adjust the parameters of the model to optimize the performance.

## 4. Custom Module

Now we will build a model of our physical system: a phase hologram illuminated by a laser beam.

PyTorch's torch.nn.Module provides a convenient way to package the design variables and the forward model together. Although nn.Module is commonly used to build neural networks, there is nothing inherently "neural" about it—we can use it to represent any differentiable model.

The essential pieces of our custom nn.Module are:

*   \_\_init__() — defines the model and its properties.
    * Should define 1 or more nn.Parameter objects, which hold trainable parameters. Our parameter will be the phase of the hologram.
    * May also define buffers, which are non-trainable tensors which are considered part of the model state. If a variable needs to be on the same device as a parameter, or if you want it to be saved when you save the model, you should define it as a buffer. We will include the amplitude distribution of the input laser beam as a buffer.
*   forward() — defines how the model calculates its output from its parameters. Our forward model will calculate the far-field diffraction pattern using an FFT.

Because the forward calculation is performed using PyTorch operations, autograd can calculate how the diffraction pattern changes with every pixel of the hologram phase. This will allow us to optimize the hologram using gradient descent.

In [ ]:
def zeropad(original, n):
    """ Increases the size of a tensor by zero padding last 2 dimensions. """
    assert original.size(-1) == original.size(-2)
    n_in = original.size(-1)
    pre = (n + n_in%2 - n_in) // 2
    post = (n + n%2 - n_in) // 2
    return F.pad(original, (pre, post, pre, post), 'constant', 0)

def unpad(original, n):
    """ Returns a view of the center (n x n) region of a torch.tensor. """
    assert original.size(-1) == original.size(-2)
    n_in = original.size(-1)
    center = slice((n_in + n%2 - n) // 2, (n_in + n%2 + n) // 2)
    return original[..., center, center]

class PhaseHologram(nn.Module):
    """
    This class calculates the far-field diffraction pattern of a phase hologram
    using the Fraunhofer diffraction equation.

    Parameters:
        phase: The phase of the hologram.

    Subclassing nn.Module is not necessary, but is good practice because it
    provides access to many useful class methods and functionality, particularly
    for complex models with many parameters and hierarchy.
    """
    def __init__(self, doe_diameter=2e-3, image_width=0.05,
                 wavelength=650e-9, device=None):
        """
        Args:
            wavelength: float, illumination wavelength, meters.
            doe_diameter: float, diameter of the hologram (diffractive optical element), meters.
            image_width: float, angular extent of holographic image, radians.
        """
        # Always call the initializer of the base class nn.Module.
        super().__init__()
        self.wavelength = wavelength

        # Sampling is important in Fourier optics.
        # I've calculated the correct sampling parameters for the desired hologram and image size.
        self.num_elements = 2 * int(np.ceil(image_width * doe_diameter / wavelength))
        self.pitch = wavelength / (2 * image_width)
        self.angular_pitch = image_width / self.num_elements

        # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

        # Define the phase as a Parameter, which is a kind of tensor that has
        # special behavior when assigned as a Module attribute.
        #  * It is automatically added to the models parameter list.
        #  * It requires gradients by default.
        phase = torch.zeros((self.num_elements, self.num_elements))
        self.phase = nn.Parameter(phase)

        # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

        # Define the input laser beam.
        # We'll use a Gaussian with a deviation equal to the radius of the DOE.
        x = self.pitch * (torch.arange(self.num_elements) - self.num_elements//2)

        r_squared = x**2 + x[:, None]**2  # Basically r^2 = x^2 + y^2

        sigma = doe_diameter / 2
        amp = torch.exp(-r_squared / (2*sigma**2))
        amp[r_squared > (doe_diameter**2)/4] = 0

        amp /= (amp ** 2).sum().sqrt()  # Normalize the power in the input beam to 1

        self.register_buffer("amp", amp)

        self.to(device=device)

    def forward(self):
        """
        Calculates the far-field holographic image formed by the hologram.
        phase -> complex field --FFT--> angular spectrum -> intensity
        """
        field = self.amp * torch.exp(1j*self.phase)
        field = zeropad(field, 2*self.num_elements)
        angular_spectrum = fft.fftshift(fft.fft2(field, norm="ortho"))
        angular_spectrum = unpad(angular_spectrum, self.num_elements)
        return torch.abs(angular_spectrum)**2

    def plot(self):
        with torch.no_grad():
            phase = self.phase.detach().clone()
            phase = (phase + np.pi) % (2 * np.pi) - np.pi
            phase[self.amp == 0] = float('nan')

            I = self.forward().detach().cpu()

        fig, ax = plt.subplots(1, 2, figsize=(8,4))

        extent = [-self.num_elements/2 * self.pitch,
                  (self.num_elements/2 - 1) * self.pitch,
                  (self.num_elements/2 - 1) * self.pitch,
                  -self.num_elements/2 * self.pitch]
        im = ax[0].imshow(phase.cpu(), cmap='hsv', extent=extent,
                          vmin=-np.pi, vmax=np.pi)
        divider = make_axes_locatable(ax[0])
        cax = divider.append_axes("right", size="5%", pad=0.05)
        cbar = fig.colorbar(im, cax=cax)
        cbar.set_ticks([-np.pi, 0, np.pi])
        cbar.set_ticklabels([r"$-\pi$", "0", r"$\pi$"])
        ax[0].set_title("Hologram Phase")
        ax[0].axis('off')

        extent = [-self.num_elements/2 * self.angular_pitch,
                  (self.num_elements/2 - 1) * self.angular_pitch,
                  (self.num_elements/2 - 1) * self.angular_pitch,
                  -self.num_elements/2 * self.angular_pitch]
        ax[1].imshow(I, cmap='hot', extent=extent)
        ax[1].set_title("Diffraction Pattern")
        ax[1].axis('off')
        return fig, ax

model = PhaseHologram(device=device)

f, ax = model.plot()
plt.show()

The hologram phase is initialized as a constant value. This is physically equivalent to an unpatterned film, i.e. a window. This should behave just like a normal laser pointer; the diffraction pattern is just a single spot.

The initial state of the parameters is often an important hyperparameter. If you have time, try changing the code so the phase is initialized as a tensor of random values between -pi and pi.

How does this change the initial state of the holographic image?
How does this affect the convergence of the optimizer?

## 5. Optimization Target

Choose the image that you want the hologram to project. **For best results, use a simple black-on-white line drawing.**

Instructions:
1. Run the cell.
2. Upload a simple line-art image.
3. To use the default goat image, cancel the file upload without selecting a file.

The image will automatically be resized without changing its aspect ratio, converted to black and white, and inverted so that the features are white and the background is black.

In [ ]:
uploaded = files.upload()
name = list(uploaded.keys())[0] if uploaded else "goat.jpg"
img = Image.open(name).convert('L')
img = ImageOps.pad(img, (model.num_elements, model.num_elements), color=255)

target = torch.tensor(np.array(img) < 128, dtype=torch.float32)

plt.imshow(target, cmap='gray')
plt.title("Target")
plt.axis('off')
plt.show()


## 6. Merit Function

Before we can optimize the hologram, we need to define what makes one design better than another.

The merit function translates our design goals into numbers that the optimizer can maximize or minimize. Choosing a good merit function is one of the most important parts of inverse design: the optimizer will optimize exactly what we ask for, which may not be exactly what we intended!

For our hologram, we will consider two objectives:

*   Efficiency — What fraction of the optical power lands inside the target region?
*   Correlation — How closely does the shape of the diffraction pattern match the target?

These objectives are related, but they are not the same. A hologram can achieve high efficiency without producing a good image, or produce a recognizable image while wasting much of the available light.

In [ ]:
def efficiency(intensity, target):
    """ Calculates the fraction of the holographic image that is on-target. """
    return (intensity*target).sum()

def correlation(intensity, target):
    """
    Calculates the normalized correlation coefficient between the
    holographic image and the target.
    """
    return (((intensity-intensity.mean())*(target-target.mean())).mean()
              /(intensity.std()*target.std()+1e-8))

If you have time, see if you can think of a different or extra merit function term that could be used to judge the quality of the hologram. Add your function here, then see what impact it has when you include it in the total loss in the training loop in section 7.

## 7. Inverse Design loop

Now we have everything we need to design our hologram!

The optimization loop follows the same basic steps as our simple gradient-descent example:

1. **Forward:** Simulate the diffraction pattern produced by the current hologram.
2. **Evaluate Merit:** Calculate how well it satisfies our design objectives.
3. **Backward:** Use autograd to calculate how the gradient of the merit function with respect to the hologram phase.
4. **Update:** Use an optimizer to modify the phase.
5. **Repeat.**

Instead of writing the gradient-descent update ourselves, we will use the **Adam optimizer**. Because the hologram phase was defined as an `nn.Parameter`, `model.parameters()` automatically tells Adam which values it should optimize.

### Your turn

Run the optimization and look at the resulting hologram and diffraction pattern.

Then experiment with the hyperparameters:

* Change **`beta`**. What happens when you optimize only correlation (`beta = 0`) or only efficiency (`beta = 1`)?
* Try changing the **learning rate**, number of iterations, or the optimizer.
* Try changing the initial state of the hologram, either here or in section 4.
* If you created your own merit-function term in Section 6, add it to `merit` and see how the result changes.

In [ ]:
# Hyperparameters -- Try changing these!
learn_rate = 0.5
beta = 0.5  # Relative weight of efficiency. 0 = correlation only; 1 = efficiency only.
max_iterations = 100

# Initialize the model
model = PhaseHologram(device=device)

# Make sure the target is on the same device as the model.
target = target.to(device)

# Define an optimizer.
optimizer = torch.optim.Adam(model.parameters(), lr=learn_rate, maximize=True)

# Initialize lists to keep track of the merit function
eff_history = []
corr_history = []

for i in range(max_iterations):
    optimizer.zero_grad()  # Resets any gradients calculated in previous iterations
    I = model()  # Forward calculation
    eff = efficiency(I, target)
    corr = correlation(I, target)
    merit = beta * eff + (1-beta) * corr  # Calculate the merit function
    merit.backward()  # Calculate the gradients
    optimizer.step()  # Update the parameters using the optimizers update rule

    eff_history.append(eff.item())  # Keep track of the merit function values in each iteration.
    corr_history.append(corr.item())

# Calculate the final merit values
with torch.no_grad():
    I = model()
    eff_history.append(efficiency(I, target).item())
    corr_history.append(correlation(I, target).item())

# Plots
_ = model.plot()

f, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.plot(eff_history, label='efficiency')
ax.plot(corr_history, label='correlation')
ax.set_ylim((0, 1))
ax.set_xlabel('Iteration')
ax.set_title("Merit")
ax.legend()
plt.show()